# 05 — Grad-CAM Validation

**Input datasets:** `lemon-dataset` + `lemon-model`

Generates Grad-CAM maps for held-out subjects and checks the model is attending
to anatomically correct brain regions.

Also produces `region_scores_example.json` — the exact format the FastAPI
backend will return to the frontend for each scan.

In [ ]:
!pip install -q tensorflow nibabel scipy scikit-learn matplotlib

In [ ]:
import numpy as np, json, os
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

DATA_DIR  = '/kaggle/input/lemon-dataset'
MODEL_DIR = '/kaggle/input/lemon-model'

model = keras.models.load_model(f'{MODEL_DIR}/best_model.keras')
X     = np.load(f'{DATA_DIR}/X.npy')
y_reg = np.load(f'{DATA_DIR}/y_regression.npy')

# Channels-last for TF
X_tf = np.transpose(X, (0, 2, 3, 4, 1)).astype(np.float32)

n_test  = max(1, len(X_tf) // 10)
X_test  = X_tf[-n_test:]
yr_test = y_reg[-n_test:]
print(f'Test subjects: {n_test}')
print('Model loaded ✓')

In [ ]:
from scipy.ndimage import zoom

def grad_cam_3d(model, volume, output_name='wellbeing_output', target_idx=0):
    """
    volume: (1, 48, 48, 32, 2)
    Returns (48, 48, 32) float32 Grad-CAM map, normalised 0–1.
    """
    # Find last Conv3D layer automatically
    last_conv = next(
        l.name for l in reversed(model.layers)
        if isinstance(l, tf.keras.layers.Conv3D)
    )

    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv).output,
                 model.get_layer(output_name).output]
    )

    with tf.GradientTape() as tape:
        inp_t = tf.cast(volume, tf.float32)
        conv_out, pred = grad_model(inp_t)
        score = pred[0, target_idx]

    grads   = tape.gradient(score, conv_out)[0]     # (h, w, d, C)
    weights = tf.reduce_mean(grads, axis=(0,1,2))   # (C,)
    cam     = tf.reduce_sum(conv_out[0] * weights, axis=-1).numpy()
    cam     = np.maximum(cam, 0)

    # Upsample to input space
    scale = np.array([48, 48, 32]) / np.array(cam.shape)
    cam   = zoom(cam, scale, order=1)
    cam   = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam.astype(np.float32)

print('Grad-CAM function ready ✓')

In [ ]:
TARGET_AFFINE = np.diag([4.0, 4.0, 4.5, 1.0])

ROIS = {
    'Amygdala (L)':        (-24, -4, -20),
    'Anterior cingulate':  (0, 28, 24),
    'Insula (R)':          (38, 2, 2),
    'vmPFC':               (0, 52, -8),
    'dlPFC (L)':           (-44, 28, 28),
    'Hippocampus (L)':     (-28, -22, -16),
    'Posterior cingulate': (-2, -52, 26),
    'Angular gyrus (L)':   (-46, -64, 28),
}

def mni_to_vox(coord, affine):
    return np.clip(
        np.round((np.linalg.inv(affine) @ [*coord, 1])[:3]).astype(int),
        0, [47, 47, 31]
    )

def score_rois(cam, radius=2):
    x, y, z = np.mgrid[:48, :48, :32]
    scores = {}
    for name, mni in ROIS.items():
        vox  = mni_to_vox(mni, TARGET_AFFINE)
        dist = np.sqrt((x-vox[0])**2+(y-vox[1])**2+(z-vox[2])**2)
        scores[name] = float(cam[dist <= radius].mean())
    return scores

print('ROI scoring ready ✓')

In [ ]:
TARGET_NAMES = ['trait_anxiety', 'chronic_stress', 'neuroticism']
all_roi_scores = []

for i in range(min(5, n_test)):
    vol   = X_test[i:i+1]
    preds = model.predict(vol, verbose=0)
    wb_pred  = preds[0][0]
    net_pred = preds[1][0]

    print(f'\n── Subject {i+1} ──')
    for j, name in enumerate(TARGET_NAMES):
        print(f'  {name:20s}  true={yr_test[i,j]:+.2f}  pred={wb_pred[j]:+.2f}')
    print(f'  Networks (DMN/SN/ECN): {net_pred.round(3)}')

    cam = grad_cam_3d(model, vol, output_name='wellbeing_output', target_idx=0)
    roi_scores = score_rois(cam)
    all_roi_scores.append(roi_scores)

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(vol[0,:,:,16,0].T, cmap='gray', origin='lower')
    axes[0].set_title('Mean BOLD')
    axes[1].imshow(vol[0,:,:,16,1].T, cmap='hot',  origin='lower')
    axes[1].set_title('fALFF')
    axes[2].imshow(vol[0,:,:,16,0].T, cmap='gray', origin='lower', alpha=0.6)
    axes[2].imshow(cam[:,:,16].T,     cmap='jet',  origin='lower', alpha=0.5)
    axes[2].set_title('Grad-CAM (trait anxiety)')
    plt.suptitle(f'Subject {i+1}')
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/gradcam_sub{i+1}.png', dpi=100)
    plt.show()

In [ ]:
# ── Mean ROI scores — this is what the backend will return per scan ───────────
mean_roi = {k: float(np.mean([s[k] for s in all_roi_scores])) for k in ROIS}
max_val  = max(mean_roi.values()) + 1e-8
norm_roi = {k: round(v/max_val, 3) for k, v in mean_roi.items()}

print('Mean Grad-CAM scores per ROI (normalised 0–1):')
for k, v in sorted(norm_roi.items(), key=lambda x: -x[1]):
    bar = '█' * int(v * 30)
    print(f'  {k:25s}  {v:.3f}  {bar}')

with open('/kaggle/working/region_scores_example.json', 'w') as f:
    json.dump(norm_roi, f, indent=2)

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

preds_all = model.predict(X_test, verbose=0)
wb_preds  = preds_all[0]

print('\nRegression performance on held-out test set:')
print(f'{"Target":25s}  MAE    R²')
print('-'*42)
for i, name in enumerate(TARGET_NAMES):
    mae = mean_absolute_error(yr_test[:,i], wb_preds[:,i])
    r2  = r2_score(yr_test[:,i], wb_preds[:,i])
    print(f'{name:25s}  {mae:.3f}  {r2:+.3f}')

print()
print('Good benchmarks for n~200 fMRI dataset:')
print('  MAE < 0.8 z-scores, R² > 0.15')
print()
print('='*55)
print('ALL NOTEBOOKS COMPLETE')
print('='*55)
print('Files to deploy with FastAPI backend:')
print('  /kaggle/input/lemon-model/best_model.keras')
print('  /kaggle/input/lemon-model/wb_stats.json')
print('  /kaggle/input/lemon-model/net_scaler.pkl')